# Refined Yearly AMR Filter

This notebook adds a second LLM screening step on top of the existing AMR-positive yearly dataset.

Current notebook-level LLM usage in this folder is:

- `2_filter.ipynb`: first-pass PMID filtering for `(pmid, pathogen, antibiotic)` using `gpt-4o-mini`
- `2_1_iterative_filter.ipynb`: the same first-pass filter with iterative batching and resume support

This notebook performs a second-pass, intent-based filter on the yearly positive set to sharpen the decision boundary between:

- real-world or applied AMR studies to keep
- laboratory-model studies to exclude

It reuses the cached title and abstract text already stored in the first-pass outputs, saves checkpoints incrementally, and rebuilds a refined yearly dataframe for downstream scientific-interest analysis.


In [1]:
%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import os

import pandas as pd

from amr.llm import create_llm_client
from amr.refined_filtering import (
    REFINED_AMR_SYSTEM_PROMPT,
    prepare_yearly_positive_candidates,
    run_refined_amr_filter,
    build_refined_yearly_positive_dataset,
)

DATA_DIR = Path('.')
YEARLY_PATH = DATA_DIR / 'pmids_results_positive_yearly.parquet'
FIRST_PASS_FILTER_PATH = DATA_DIR / 'pmids_results_filtered.parquet'
ABSTRACT_CACHE_PATH = DATA_DIR / 'pmid_abstracts.parquet'

REFINED_SCREEN_PATH = DATA_DIR / 'pmids_results_positive_yearly_refined_screen.parquet'
REFINED_PAIR_PMID_PATH = DATA_DIR / 'pmids_results_positive_yearly_refined_pair_pmid.parquet'
REFINED_YEARLY_PATH = DATA_DIR / 'pmids_results_positive_yearly_refined.parquet'
LOG_PATH = DATA_DIR / 'pmids_results_positive_yearly_refined.log'

LLM_PROVIDER = 'openai'
MODEL_NAME = 'gpt-4o-mini'
SAVE_EVERY = 50
PROGRESS_EVERY = 25
MAX_LLM_CALLS = None
DRY_RUN = False
VERBOSE = 1

llm_kwargs = {}
if MODEL_NAME:
    llm_kwargs['model'] = MODEL_NAME

llm = create_llm_client(provider=LLM_PROVIDER, **llm_kwargs)
print(f'Using provider={LLM_PROVIDER}, model={MODEL_NAME or "provider default"}, dry_run={DRY_RUN}, max_llm_calls={MAX_LLM_CALLS}')


Using provider=openai, model=gpt-4o-mini, dry_run=False, max_llm_calls=None


In [2]:
yearly_df = pd.read_parquet(YEARLY_PATH)
filtered_df = pd.read_parquet(FIRST_PASS_FILTER_PATH)
abstract_cache_df = pd.read_parquet(ABSTRACT_CACHE_PATH) if ABSTRACT_CACHE_PATH.exists() else None

candidate_df = prepare_yearly_positive_candidates(
    yearly_df,
    filtered_df,
    abstract_cache_df=abstract_cache_df,
)

if REFINED_SCREEN_PATH.exists():
    reviewed_df = pd.read_parquet(REFINED_SCREEN_PATH)
    reviewed_keys = set(
        zip(
            reviewed_df['pmid'].astype(str),
            reviewed_df['pathogen'].astype(str),
            reviewed_df['antibiotic'].astype(str),
            reviewed_df['year'].astype(int),
        )
    )
else:
    reviewed_df = None
    reviewed_keys = set()

candidate_keys = set(
    zip(
        candidate_df['pmid'].astype(str),
        candidate_df['pathogen'].astype(str),
        candidate_df['antibiotic'].astype(str),
        candidate_df['year'].astype(int),
    )
)
remaining = len(candidate_keys - reviewed_keys)
missing_abstract_n = int(candidate_df['abstract'].fillna('').eq('').sum())

print(f'yearly positive rows: {len(yearly_df):,}')
print(f'second-pass candidates: {len(candidate_df):,}')
print(f'already screened: {len(reviewed_keys):,}')
print(f'remaining to screen: {remaining:,}')
print(f'candidates without abstract text: {missing_abstract_n:,}')

candidate_df.head()


yearly positive rows: 45,244
second-pass candidates: 154,906
already screened: 0
remaining to screen: 154,906
candidates without abstract text: 0


,pmid,pathogen,antibiotic,year,first_pass_decision,first_pass_rationale,title,abstract
0,1040755,Aeromonas hydrophila,Ampicillin,1975,yes,Strains of Aeromonas hydrophila were mostly re...,Antibiotic sensitivity of two Aeromonas and ni...,An agar dilution method was used to determine ...
1,1040755,Aeromonas hydrophila,Carbenicillin,1975,yes,Strains of Aeromonas hydrophila were mostly re...,Antibiotic sensitivity of two Aeromonas and ni...,An agar dilution method was used to determine ...
2,1041215,Bacteroides fragilis,Carbenicillin,1975,yes,60% of strains were susceptible to carbenicill...,In vitro activity of penicillins against anaer...,The in vitro susceptibility of 162 anaerobic i...
3,804023,Bacteroides fragilis,Carbenicillin,1975,yes,The abstract states that five of 123 strains o...,Susceptibility of Anaerobic bacteria to carben...,The agar dilution technique was used for deter...
4,1179177,Bacteroides fragilis,Doxycycline,1975,yes,The abstract states that the strain of Bactero...,"Some clinical, immunological and bacteriologic...",A case of recurrent pyogenic arthritis in a kn...


## Refined second-pass prompt

The second-pass screen is intent-based rather than keyword-based.

- `YES`: real-world or applied AMR context, including molecular characterisation of naturally occurring resistance
- `NO`: the strain is primarily a laboratory model or experimental platform and AMR is incidental to that setup

The strict system prompt used below is stored in `amr.refined_filtering.REFINED_AMR_SYSTEM_PROMPT`.


In [3]:
print(REFINED_AMR_SYSTEM_PROMPT)


You are screening PubMed abstracts for relevance to antimicrobial resistance (AMR) in bacterial pathogens.

The abstract already mentions antimicrobial resistance in a bacterial pathogen species.

Decision rule:
- Answer NO if the study primarily uses the bacterial strain as a laboratory model system (for example a pharmacology assay platform, molecular biology tool, evolutionary laboratory experiment, or genetic engineering system), and the focus is not on real-world antimicrobial resistance in clinical, environmental, veterinary, food, or epidemiological contexts.
- Answer YES if the study investigates antimicrobial resistance in a real-world or applied context, including clinical infections, patient isolates, outbreaks, surveillance, epidemiology, environmental or ecological AMR studies, food microbiology, veterinary or agricultural settings, or characterisation of resistance phenotypes or genotypes in naturally occurring strains.
- If a study describes resistance in a clinical or r

In [ ]:
%%time

refined_df = run_refined_amr_filter(
    candidate_df,
    llm,
    output_path=str(REFINED_SCREEN_PATH),
    resume_from=reviewed_df,
    save_every=SAVE_EVERY,
    progress_every=PROGRESS_EVERY,
    max_llm_calls=MAX_LLM_CALLS,
    dry_run=DRY_RUN,
    verbose=VERBOSE,
    log_path=str(LOG_PATH),
)

refined_df['refined_decision'].value_counts(dropna=False)


Processed     25/154906 yes:    19 no:     6 uncertain:     0
Processed     50/154906 yes:    28 no:    22 uncertain:     0
Processed     75/154906 yes:    35 no:    40 uncertain:     0
Processed    100/154906 yes:    41 no:    59 uncertain:     0
Processed    125/154906 yes:    58 no:    67 uncertain:     0
Processed    150/154906 yes:    76 no:    74 uncertain:     0
Processed    175/154906 yes:    91 no:    84 uncertain:     0
Processed    200/154906 yes:   107 no:    93 uncertain:     0
Processed    225/154906 yes:   121 no:   104 uncertain:     0
Processed    250/154906 yes:   133 no:   117 uncertain:     0
Processed    275/154906 yes:   151 no:   124 uncertain:     0
Processed    300/154906 yes:   171 no:   129 uncertain:     0
Processed    325/154906 yes:   196 no:   129 uncertain:     0
Processed    350/154906 yes:   219 no:   131 uncertain:     0
Processed    375/154906 yes:   235 no:   140 uncertain:     0
Processed    400/154906 yes:   257 no:   143 uncertain:     0
Processe

In [ ]:
refined_pair_pmid_df, refined_yearly_df = build_refined_yearly_positive_dataset(
    yearly_df,
    refined_df,
    output_path=str(REFINED_YEARLY_PATH),
    pair_output_path=str(REFINED_PAIR_PMID_PATH),
)

print(f'saved refined screening decisions to {REFINED_SCREEN_PATH}')
print(f'saved refined pair-PMID table to {REFINED_PAIR_PMID_PATH}')
print(f'saved refined yearly counts to {REFINED_YEARLY_PATH}')
print()
print(f'refined kept pair-PMID rows: {len(refined_pair_pmid_df):,}')
print(f'refined yearly rows: {len(refined_yearly_df):,}')
print(f'original yearly total count: {int(yearly_df["count"].sum()):,}')
print(f'refined yearly total count: {int(refined_yearly_df["count"].sum()) if not refined_yearly_df.empty else 0:,}')

refined_yearly_df.head()


In [ ]:
comparison_df = (
    yearly_df[['pathogen', 'antibiotic', 'year', 'count']]
    .rename(columns={'count': 'original_count'})
    .merge(
        refined_yearly_df[['pathogen', 'antibiotic', 'year', 'count']].rename(columns={'count': 'refined_count'}),
        on=['pathogen', 'antibiotic', 'year'],
        how='left',
    )
)
comparison_df['refined_count'] = comparison_df['refined_count'].fillna(0).astype(int)
comparison_df['removed_count'] = comparison_df['original_count'] - comparison_df['refined_count']

comparison_df.sort_values(['removed_count', 'original_count'], ascending=[False, False]).head(20)
